# Notebook 18: Tabular Representations

## Why This Matters
Most enterprise data is tabular — transactions, customer records, medical records, sensor logs. Transformer-based representation learning (TabNet, SAINT) has closed much of the gap with gradient boosting on tabular data, and embeddings for categorical features transfer the same retrieval, clustering, and drift detection ideas from text and vision to structured data.

## Structure
1. Why tabular data is hard for neural networks (narrative)
2. Entity embeddings for categoricals
3. TabNet — attention-based tabular learning
4. Feature embeddings for retrieval and similarity
5. Bridge to self-supervised tabular and time series

## What You'll Learn
- Why entity embeddings improve on one-hot encoding for categoricals
- How TabNet uses sequential attention to select features
- How to use tabular embeddings for similarity search and anomaly detection


## Background

### Why tabular data is hard for neural networks

Gradient boosted trees (XGBoost, LightGBM) remain competitive or superior on most tabular benchmarks. The reasons:
- Tabular features are heterogeneous in scale and distribution
- Missing values and outliers are common
- Feature interactions are often local (tree splits) not global (attention)
- Datasets are often small — no pretraining benefit
- Categorical features require special handling

Neural networks address this with:
- **Entity embeddings** — learn a dense vector per category level, replacing one-hot
- **Normalization** — batch norm, layer norm, or quantile normalization per feature
- **Attention** — TabNet and SAINT use attention to select relevant features per sample

The practical recommendation: use XGBoost/LightGBM for prediction tasks. Use neural embeddings when you need a vector representation (retrieval, clustering, transfer).

### Entity embeddings

For a categorical feature with K levels, instead of a K-dimensional one-hot vector, learn a d-dimensional embedding per level (d << K). The embedding is trained end-to-end with the task.

Why it works:
- Similar categories end up nearby — the model learns that "Monday" and "Tuesday" are more similar than "Monday" and "Saturday"
- The embedding dimension is a hyperparameter — typically d = min(50, ceil(K/2))
- Reduces input dimensionality for high-cardinality features

Used in: customer churn models, recommendation systems, fraud detection.


In [ ]:
# %pip install -q torch numpy scikit-learn matplotlib pandas

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score

torch.manual_seed(42)
np.random.seed(42)
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 1. Entity Embeddings for Categorical Features

In [ ]:
# Simulate a dataset with mixed features
np.random.seed(42)
N = 2000
n_cat_levels = [10, 5, 8, 15]  # 4 categorical features with varying cardinality
n_num = 5                        # 5 numerical features

# Generate synthetic data
cat_data = np.column_stack([np.random.randint(0, k, N) for k in n_cat_levels])
num_data = np.random.randn(N, n_num)
labels   = ((num_data[:, 0] + cat_data[:, 0] * 0.1 + np.random.randn(N) * 0.5) > 0).astype(int)

print(f"Dataset: {N} samples, {len(n_cat_levels)} categorical + {n_num} numerical features")
print(f"Categorical levels: {n_cat_levels}")
print(f"Label balance: {labels.mean():.2f}")


class TabularEmbeddingModel(nn.Module):
    """Tabular model with entity embeddings for categoricals."""
    def __init__(self, cat_cardinalities, n_num, embed_dim=None, hidden=64, n_out=2):
        super().__init__()
        # Entity embeddings: one embedding table per categorical feature
        if embed_dim is None:
            # Rule of thumb: min(50, ceil(K/2))
            embed_dims = [min(50, (k + 1) // 2) for k in cat_cardinalities]
        else:
            embed_dims = [embed_dim] * len(cat_cardinalities)

        self.embeddings = nn.ModuleList([
            nn.Embedding(k, d) for k, d in zip(cat_cardinalities, embed_dims)
        ])
        total_embed_dim = sum(embed_dims)
        self.input_dim  = total_embed_dim + n_num

        self.mlp = nn.Sequential(
            nn.Linear(self.input_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, n_out),
        )

    def get_embedding(self, cat_x: torch.Tensor, num_x: torch.Tensor) -> torch.Tensor:
        """Return the pre-classification embedding (for retrieval/clustering)."""
        cat_embs = [emb(cat_x[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat(cat_embs + [num_x], dim=1)
        # Return intermediate representation (before final classifier)
        return self.mlp[:4](x)  # through first hidden + BN + ReLU + Dropout

    def forward(self, cat_x, num_x):
        cat_embs = [emb(cat_x[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat(cat_embs + [num_x], dim=1)
        return self.mlp(x)


model_tab = TabularEmbeddingModel(n_cat_levels, n_num, hidden=64).to(device)
print(f"\nModel parameters: {sum(p.numel() for p in model_tab.parameters()):,}")
print(f"Input dim (after embedding): {model_tab.input_dim}")

# Training
from torch.utils.data import TensorDataset, DataLoader, random_split

X_cat = torch.tensor(cat_data, dtype=torch.long)
X_num = torch.tensor(StandardScaler().fit_transform(num_data), dtype=torch.float32)
y     = torch.tensor(labels, dtype=torch.long)

dataset = TensorDataset(X_cat, X_num, y)
n_train = int(0.8 * N)
train_ds, val_ds = random_split(dataset, [n_train, N - n_train])
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=256)

optimizer = optim.Adam(model_tab.parameters(), lr=1e-3, weight_decay=1e-4)
for epoch in range(30):
    model_tab.train()
    for xc, xn, yb in train_loader:
        xc, xn, yb = xc.to(device), xn.to(device), yb.to(device)
        loss = F.cross_entropy(model_tab(xc, xn), yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

model_tab.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for xc, xn, yb in val_loader:
        preds = model_tab(xc.to(device), xn.to(device)).argmax(1).cpu()
        all_preds.extend(preds.tolist()); all_true.extend(yb.tolist())
print(f"Validation accuracy: {accuracy_score(all_true, all_preds):.3f}")

In [ ]:
# Visualize what the entity embedding learned for one categorical feature
emb_weights = model_tab.embeddings[0].weight.detach().cpu().numpy()  # [K, d]

print(f"Category 0 embedding: {emb_weights.shape} (K levels × {emb_weights.shape[1]} dims)")
print()

from sklearn.decomposition import PCA
if emb_weights.shape[1] >= 2:
    pca = PCA(n_components=2).fit_transform(emb_weights)
    plt.figure(figsize=(6, 5))
    plt.scatter(pca[:, 0], pca[:, 1], c=range(len(pca)), cmap='tab10', s=100)
    for i in range(len(pca)):
        plt.annotate(f"cat_{i}", pca[i], fontsize=9, ha='center', va='bottom',
                     xytext=(0, 5), textcoords='offset points')
    plt.title(f"Entity embeddings — Category 0 ({n_cat_levels[0]} levels)")
    plt.xlabel("PC1"); plt.ylabel("PC2")
    plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
    print("Similar categories cluster together — the model learned ordinal structure.")

## 2. TabNet (Narrative + Usage)

TabNet (Arik & Pfister, 2020) uses sequential multi-step attention to select which features to use at each decision step — interpretable by design.

```python
from pytorch_tabnet.tab_model import TabNetClassifier

clf = TabNetClassifier(
    n_d=16, n_a=16,          # width of decision step
    n_steps=3,                # number of sequential steps
    gamma=1.3,                # sparsity regularization
    cat_idxs=[0, 1, 2, 3],  # categorical feature indices
    cat_dims=[10, 5, 8, 15], # cardinality of each categorical
    cat_emb_dim=4,            # embedding dim per categorical
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    mask_type='sparsemax',
)
clf.fit(X_train, y_train, eval_set=[(X_val, y_val)])

# Feature importances — what TabNet attended to
importances = clf.feature_importances_

# Embeddings for retrieval
embeddings = clf.predict_proba(X_test)  # use penultimate layer for embeddings
```

TabNet's main advantage over vanilla MLP: built-in feature importance via attention masks. Practical disadvantage: slower to train and harder to tune than XGBoost.


## 3. Bridge to Self-Supervised Tabular and Time Series

Entity embeddings and TabNet are supervised — they need labels. For unsupervised or semi-supervised settings (most large enterprise datasets), self-supervised methods adapt the contrastive learning ideas from vision:

**SCARF** (Bahri et al., 2021): apply random feature masking as augmentation (analogous to image cropping), train with NT-Xent loss. The result: embeddings that cluster similar rows even without labels.

**TS2Vec** (Yue et al., 2022): contrastive learning for time series — treat two temporal crops of the same series as positive pairs.

Notebook 19 implements these, completing the picture of how contrastive representation learning generalizes from text (SBERT) → vision (SimCLR) → tabular (SCARF) → time series (TS2Vec).


## Summary

| Method | Type | Best for |
|--------|------|----------|
| Entity embeddings | Supervised | Categorical features in any tabular task |
| TabNet | Supervised | Interpretable feature selection |
| SAINT | Supervised | Inter-feature attention (transformers for tables) |
| SCARF | Self-supervised | Unlabeled tabular data, clustering |

**Rule of thumb:** entity embeddings are always worth adding for high-cardinality categoricals. Use TabNet when feature interpretability matters. Use XGBoost when accuracy is the only goal.

**Next:** Notebook 19 — Self-Supervised Tabular & Time Series. SCARF and TS2Vec: contrastive pretraining on structured data without labels.
